# Foil sections

Get and set **control points**, **geometric parameters** (tc, camber, LE radius, ...)
and **structural parameters** (area, Ixx, ...) of a foil section.

Open a project in the GUI first — the SDK works on the same session.

In [ ]:
from pytakeoff import TakeoffClient

# Ran `python -m pytakeoff`? Leave this line exactly as it is: the all-x
# placeholder counts as no key, so your saved credentials (or the
# TAKEOFF_API_KEY env var) are used automatically.
# Otherwise, paste your own key here - GUI: Account -> API Keys.
API_KEY = "tk_xxxxxxxx_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

client = TakeoffClient(api_key=API_KEY)
print(f"Connected to {client.base_url} as {client.username}")

project = client.projects.current() or client.projects.open(client.projects.list()[0]["name"])
project

## Pick a section

In [ ]:
import matplotlib.pyplot as plt

section = project.foil_sections()[0]
section

## The shape: outline and control points

In [ ]:
cp = section.control_points()          # {"upper", "lower", "degree", "n_coefs"}
xy = section.points()                  # computed outline

plt.figure(figsize=(9, 3))
plt.plot(*zip(*xy), "-", label="outline")
plt.plot(*zip(*cp["upper"]), "o--", label="upper control points")
plt.plot(*zip(*cp["lower"]), "s--", label="lower control points")
plt.axis("equal"); plt.legend(); plt.title(section.name)
plt.show()

## Geometric & structural parameters

In [ ]:
print("geometry: ", section.geometry())    # tc / camber in % of chord
print("structure:", section.structure())

## Set a parameter — make it 20% thicker

Parametric setters refit the B-spline and return the achieved values.

In [ ]:
geo = section.geometry()
outline_before = section.points()

section.set_geometry(tc=geo["tc"] * 1.2)

plt.figure(figsize=(9, 3))
plt.plot(*zip(*outline_before), label=f"before  (tc={geo['tc']:.2f}%)")
plt.plot(*zip(*section.points()), label=f"after   (tc={section.geometry()['tc']:.2f}%)")
plt.axis("equal"); plt.legend(); plt.title("set_geometry(tc=...)")
plt.show()

## Restore the original shape exactly

Setting the saved control points back is an exact restore (nothing was saved to disk).

In [ ]:
section.set_control_points(upper=cp["upper"], lower=cp["lower"])
print("tc back to", round(section.geometry()["tc"], 3), "%")
client.close()